<a href="https://colab.research.google.com/github/veenanayak944-bot/Project-based-on-LLM-Transformer/blob/main/Recipe_Assistent_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers transformers accelerate

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
recipes = [
    {
        "title": "Garlic Butter Chicken",
        "ingredients": "chicken breast, butter, garlic, salt, pepper, parsley",
        "instructions": "Season chicken with salt and pepper. Melt butter in a pan, add minced garlic, cook chicken 6-7 minutes each side, garnish with parsley."
    },
    {
        "title": "Spinach and Chickpea Curry",
        "ingredients": "spinach, chickpeas, onion, tomato, garlic, ginger, cumin, turmeric",
        "instructions": "Saute onion, garlic and ginger. Add tomato and spices, cook 5 minutes. Add chickpeas and spinach, simmer 10 minutes."
    },
    {
        "title": "Classic Pancakes",
        "ingredients": "flour, milk, eggs, sugar, baking powder, butter",
        "instructions": "Mix dry ingredients. Whisk in milk and eggs. Cook spoonfuls of batter on a buttered griddle until bubbles form, flip and cook other side."
    },
    {
        "title": "Tomato Basil Pasta",
        "ingredients": "pasta, tomatoes, garlic, basil, olive oil, parmesan",
        "instructions": "Cook pasta. Saute garlic in olive oil, add chopped tomatoes, simmer 10 minutes. Toss with pasta, top with basil and parmesan."
    },
    {
        "title": "Chicken Spinach Stir Fry",
        "ingredients": "chicken breast, spinach, soy sauce, garlic, ginger, sesame oil",
        "instructions": "Stir fry chicken in sesame oil until cooked. Add garlic and ginger, then spinach and soy sauce, cook until wilted."
    }
]

In [7]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
recipe_texts = [f"Title: {r['title']}. Ingredients: {r['ingredients']}. Instructions: {r['instructions']}" for r in recipes]
recipe_embeddings = embedder.encode(recipe_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
def retrieve(query, top_k=2):
    query_embedding = embedder.encode([query])
    similarities = np.dot(recipe_embeddings, query_embedding.T).flatten()
    similarities = similarities / (
        np.linalg.norm(recipe_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [recipe_texts[i] for i in top_indices]

In [9]:
retrieve("what can I cook with chicken and spinach")

['Title: Chicken Spinach Stir Fry. Ingredients: chicken breast, spinach, soy sauce, garlic, ginger, sesame oil. Instructions: Stir fry chicken in sesame oil until cooked. Add garlic and ginger, then spinach and soy sauce, cook until wilted.',
 'Title: Spinach and Chickpea Curry. Ingredients: spinach, chickpeas, onion, tomato, garlic, ginger, cumin, turmeric. Instructions: Saute onion, garlic and ginger. Add tomato and spices, cook 5 minutes. Add chickpeas and spinach, simmer 10 minutes.']

In [11]:
generator = pipeline("text-generation", model="google/flan-t5-base")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

In [12]:
def recipe_assistant(query):
    context = "\n\n".join(retrieve(query))
    prompt = f"""Answer the question using only the recipes below.

{context}

Question: {query}
Answer:"""
    response = generator(prompt, max_length=200)
    return response[0]['generated_text']

In [17]:
print(recipe_assistant("What can I cook with chicken?"))
print(recipe_assistant("What can I cook with vegtables?"))
print(recipe_assistant("What can I cook with chicken and spinach?"))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the recipes below.

Title: Garlic Butter Chicken. Ingredients: chicken breast, butter, garlic, salt, pepper, parsley. Instructions: Season chicken with salt and pepper. Melt butter in a pan, add minced garlic, cook chicken 6-7 minutes each side, garnish with parsley.

Title: Chicken Spinach Stir Fry. Ingredients: chicken breast, spinach, soy sauce, garlic, ginger, sesame oil. Instructions: Stir fry chicken in sesame oil until cooked. Add garlic and ginger, then spinach and soy sauce, cook until wilted.

Question: What can I cook with chicken?
Answer:


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the recipes below.

Title: Spinach and Chickpea Curry. Ingredients: spinach, chickpeas, onion, tomato, garlic, ginger, cumin, turmeric. Instructions: Saute onion, garlic and ginger. Add tomato and spices, cook 5 minutes. Add chickpeas and spinach, simmer 10 minutes.

Title: Chicken Spinach Stir Fry. Ingredients: chicken breast, spinach, soy sauce, garlic, ginger, sesame oil. Instructions: Stir fry chicken in sesame oil until cooked. Add garlic and ginger, then spinach and soy sauce, cook until wilted.

Question: What can I cook with vegtables?
Answer:
Answer the question using only the recipes below.

Title: Chicken Spinach Stir Fry. Ingredients: chicken breast, spinach, soy sauce, garlic, ginger, sesame oil. Instructions: Stir fry chicken in sesame oil until cooked. Add garlic and ginger, then spinach and soy sauce, cook until wilted.

Title: Spinach and Chickpea Curry. Ingredients: spinach, chickpeas, onion, tomato, garlic, ginger, cumin, turmeric. Inst